# 04. GDAG-GNN & Uncertainty-Aware Price Prediction (Exp 3 & Exp 4)
**Project:** *Does Bangalore Traffic Explain House Prices? Uncertainty-Aware Price Prediction with Road-Network GNNs and Bayesian Heads*  
**Paper Grounding:** Ravichandran et al. (*Nature Scientific Reports*, 2025, 15:41208)  
**Methods:**
- **Exp 3**: Gaussian Dual Adjacency GNN (`GDAGSpatialEncoder`) + Tabular Feature Fusion.
- **Exp 4**: Uncertainty Quantification (Monte Carlo Dropout, Deep Ensemble, Heteroscedastic NLL, Conformal Prediction CQR).

---
### Objectives:
1. Train the hybrid GDAG-GNN model end-to-end in PyTorch.
2. Evaluate point prediction accuracy vs tabular baselines.
3. Quantify spatial uncertainty on unseen localities (Leave-Locality-Out splits).
4. Calibrate prediction intervals using Conformal Prediction (10% nominal miscoverage guarantee).
5. Compute the inferential economic price penalty per 10 minutes of peak commute.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import TensorDataset, DataLoader

from src.logger import ServerLogger
from src.gdag_spatial_graph import GaussianDualAdjacencyGraph
from src.models import HybridGDAGHousingModel, ConformalPredictionCalibrator
from src.evaluation import compute_point_metrics, compute_uncertainty_metrics

logger = ServerLogger.get_logger()
logger.info("Notebook 04: GDAG-GNN & Bayesian Uncertainty initialized.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {DEVICE}")

## 1. Load Data Assets & Construct Normalized DAM

In [ ]:
df = pd.read_csv('../data/bangalore_housing_spatial_commute_dataset.csv')
df_nodes = pd.read_csv('../data/bangalore_locality_graph_nodes.csv')
df_edges = pd.read_csv('../data/bangalore_locality_graph_edges.csv')

# Build Dual Adjacency Matrix
gdag = GaussianDualAdjacencyGraph(sigma=1.2, dynamic_threshold=0.30)
DAM, norm_DAM = gdag.build_dual_adjacency_matrix(df_nodes, df_edges)
norm_dam_tensor = torch.tensor(norm_DAM, dtype=torch.float32).to(DEVICE)

# Node features
gnn_node_cols = [
    'latitude', 'longitude', 'node_degree',
    'commute_whitefield_peak_min', 'commute_bellandur_peak_min',
    'commute_ecity_peak_min', 'commute_manyata_peak_min',
    'commute_mgroad_peak_min', 'avg_congestion_ratio',
    'dist_to_metro_km', 'commute_metro_access_min'
]
node_feat_raw = df_nodes[gnn_node_cols].values.astype(np.float32)
node_feat_norm = (node_feat_raw - np.mean(node_feat_raw, axis=0)) / (np.std(node_feat_raw, axis=0) + 1e-6)
node_feat_tensor = torch.tensor(node_feat_norm, dtype=torch.float32).to(DEVICE)

# Tabular & Commute features
tabular_features = ['bhk', 'total_sqft_cleaned', 'bath_cleaned', 'balcony_cleaned', 'is_ready_to_move']
area_dummies = pd.get_dummies(df['area_type'], prefix='area', drop_first=True)
df = pd.concat([df, area_dummies], axis=1)
tabular_features.extend(list(area_dummies.columns))

commute_features = [
    'commute_whitefield_peak_min', 'commute_whitefield_offpeak_min',
    'commute_bellandur_orr_peak_min', 'commute_bellandur_orr_offpeak_min',
    'commute_electronic_city_peak_min', 'commute_electronic_city_offpeak_min',
    'commute_manyata_tech_park_peak_min', 'commute_manyata_tech_park_offpeak_min',
    'commute_mg_road_cbd_peak_min', 'commute_mg_road_cbd_offpeak_min',
    'min_commute_peak_min', 'avg_commute_peak_min',
    'avg_congestion_ratio', 'dist_to_metro_km', 'commute_metro_access_min'
]

# Locality mapping
loc_map = {row['canonical_locality_name']: int(row['locality_id']) for _, row in df_nodes.iterrows()}
df['locality_node_id'] = df['canonical_locality_name'].map(loc_map)
df = df[df['locality_node_id'].notna()].copy()
df['locality_node_id'] = df['locality_node_id'].astype(int)

# Standardize inputs
X_tab = (df[tabular_features].values - np.mean(df[tabular_features].values, axis=0)) / (np.std(df[tabular_features].values, axis=0) + 1e-6)
X_comm = (df[commute_features].values - np.mean(df[commute_features].values, axis=0)) / (np.std(df[commute_features].values, axis=0) + 1e-6)
loc_ids = df['locality_node_id'].values
y = df['log_price'].values.astype(np.float32)

## 2. Train Hybrid GDAG-GNN Model (Exp 3)

In [ ]:
train_mask = (df['split_random'] == 'train').values
val_mask = (df['split_random'] == 'val').values
test_mask = (df['split_random'] == 'test').values

model = HybridGDAGHousingModel(
    tabular_dim=len(tabular_features),
    gnn_node_dim=len(gnn_node_cols),
    commute_dim=len(commute_features),
    locality_emb_dim=32,
    hidden_dim=64,
    mc_dropout=0.20
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.004, weight_decay=1e-4)
criterion = torch.nn.MSELoss()

train_ds = TensorDataset(
    torch.tensor(X_tab[train_mask], dtype=torch.float32).to(DEVICE),
    torch.tensor(X_comm[train_mask], dtype=torch.float32).to(DEVICE),
    torch.tensor(loc_ids[train_mask], dtype=torch.long).to(DEVICE),
    torch.tensor(y[train_mask], dtype=torch.float32).to(DEVICE)
)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

losses = []
model.train()
for epoch in range(35):
    ep_loss = 0.0
    for bx_tab, bx_comm, b_loc, by in train_loader:
        optimizer.zero_grad()
        mu, _ = model(bx_tab, bx_comm, b_loc, node_feat_tensor, norm_dam_tensor)
        loss = criterion(mu, by)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item() * len(by)
    losses.append(ep_loss / len(train_ds))
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}/35: Loss = {losses[-1]:.4f}")

plt.figure(figsize=(7, 3.5))
plt.plot(losses, color='crimson')
plt.title("GDAG-GNN Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.tight_layout()
plt.show()

## 3. Exp 4: Monte Carlo Dropout Uncertainty Quantification

In [ ]:
test_bx_tab = torch.tensor(X_tab[test_mask], dtype=torch.float32).to(DEVICE)
test_bx_comm = torch.tensor(X_comm[test_mask], dtype=torch.float32).to(DEVICE)
test_b_loc = torch.tensor(loc_ids[test_mask], dtype=torch.long).to(DEVICE)

# 30 stochastic forward passes with dropout active
mc_mean, mc_std = model.predict_mc_dropout(
    test_bx_tab, test_bx_comm, test_b_loc, node_feat_tensor, norm_dam_tensor, n_samples=30
)
y_test = y[test_mask]

pt_metrics = compute_point_metrics(y_test, mc_mean)
uq_metrics = compute_uncertainty_metrics(y_test, mc_mean, mc_std)

print("MC Dropout Evaluation:")
print(f"  RMSE:        {pt_metrics['rmse']:.4f}")
print(f"  MAPE:        {pt_metrics['mape_pct']:.2f}%")
print(f"  NLL:         {uq_metrics['nll']:.4f}")
print(f"  90% PICP:    {uq_metrics['picp_90_pct']:.2f}% (Nominal: 90%)")
print(f"  MPIW:        {uq_metrics['mpiw']:.4f}")

## 4. Exp 4: Conformal Prediction (CQR Calibrated Intervals)

In [ ]:
# Calibrate on validation split
model.eval()
with torch.no_grad():
    val_bx_tab = torch.tensor(X_tab[val_mask], dtype=torch.float32).to(DEVICE)
    val_bx_comm = torch.tensor(X_comm[val_mask], dtype=torch.float32).to(DEVICE)
    val_b_loc = torch.tensor(loc_ids[val_mask], dtype=torch.long).to(DEVICE)
    val_mu, _ = model(val_bx_tab, val_bx_comm, val_b_loc, node_feat_tensor, norm_dam_tensor)
    val_mu = val_mu.cpu().numpy()

calibrator = ConformalPredictionCalibrator(alpha=0.10)
q_hat = calibrator.calibrate(y[val_mask], val_mu)
lower_conf, upper_conf = calibrator.predict_interval(mc_mean)

conformal_coverage = np.mean((y_test >= lower_conf) & (y_test <= upper_conf)) * 100.0
print(f"Conformal Prediction Interval Half-Width: +/- {q_hat:.4f}")
print(f"Empirical Coverage on Test Set:           {conformal_coverage:.2f}% (Guaranteed >= 90.0%)")

## 5. Calibration Plot & Uncertainty Intervals

In [ ]:
sample_idx = np.argsort(mc_mean)[:60]
plt.figure(figsize=(14, 5))
plt.plot(range(60), y_test[sample_idx], 'ro', label='True Log-Price', markersize=4)
plt.plot(range(60), mc_mean[sample_idx], 'b-', label='Predicted Mean (mu)')
plt.fill_between(range(60), lower_conf[sample_idx], upper_conf[sample_idx], color='blue', alpha=0.15, label='90% Conformal Interval')
plt.title("Calibrated Conformal Prediction Intervals (Sample of 60 Test Listings)", fontsize=12)
plt.xlabel("Listing Index (Sorted by Price)")
plt.ylabel("log(Price in Lakhs)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Inferential Commute Elasticity (% Price Drop per 10 Min Commute)

In [ ]:
# Estimate marginal price elasticity per 10 minutes of commute to Whitefield
min_per_10 = 10.0
# Baseline prediction
with torch.no_grad():
    base_mu, _ = model(test_bx_tab[:100], test_bx_comm[:100], test_b_loc[:100], node_feat_tensor, norm_dam_tensor)
    
    # Perturb Whitefield commute by +10 min (scaled)
    wf_idx = commute_features.index('commute_whitefield_peak_min')
    perturbed_comm = test_bx_comm[:100].clone()
    std_wf = np.std(df['commute_whitefield_peak_min'])
    perturbed_comm[:, wf_idx] += (10.0 / std_wf)
    
    pert_mu, _ = model(test_bx_tab[:100], perturbed_comm, test_b_loc[:100], node_feat_tensor, norm_dam_tensor)
    
pct_change = (torch.exp(pert_mu) - torch.exp(base_mu)) / torch.exp(base_mu) * 100.0
mean_pct_change = pct_change.mean().item()
print(f"Marginal Commute Impact: {mean_pct_change:.2f}% price change per +10 min peak commute to Whitefield.")
logger.info("Notebook 04 execution completed.")